In [ ]:
!pip install -q numpy torch datasets transformers seqeval tqdm gliner

In [2]:
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from datasets import load_dataset
from transformers import (
    BertTokenizerFast,
    AutoTokenizer,
    AutoModel,
    get_cosine_schedule_with_warmup,
)
from seqeval.metrics import classification_report, f1_score
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

import warnings

warnings.filterwarnings("ignore")
DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)

print(f"Using device: {DEVICE}")

Using device: mps


---
# 1) NER Foundations

Named Entity Recognition assigns a label to each token in a sentence. The most common
encoding is **BIO** (Begin / Inside / Outside):

| Token | Label |
|-------|-------|
| Sharon | B-PER |
| Floyd | I-PER |
| flew | O |
| to | O |
| Miami | B-LOC |
| on | O |
| Friday | B-MISC |

**Why BIO over IO?** BIO disambiguates consecutive entities of the same type.
For example, `[B-PER, I-PER, B-PER]` encodes two distinct person entities,
whereas IO tagging (`[I-PER, I-PER, I-PER]`) would merge them.

**Sample input:** `["Sharon", "Floyd", "flew", "to", "Miami"]`
**Sample output:** `["B-PER", "I-PER", "O", "O", "B-LOC"]`

## 1.1 CoNLL Dataset

We use CoNLL-2003 (Tjong Kim Sang & De Meulder, 2003)

The standard NER benchmark with 4 entity types: PER, LOC, ORG, MISC.

In [3]:
from datasets import load_dataset

# tner/conll2003 and the canonical conll2003 repo are both loading scripts,
# which datasets>=3.0 no longer runs. This mirror ships the same data as
# parquet, so label names come from the schema instead of being hardcoded.
dataset = load_dataset("tomaarsen/conll2003")
# 4K of the 14K training sentences and 1K validation sentences keep the BERT epoch to a few
# minutes; production fine-tunes on the full train split for 3 epochs
dataset["train"] = dataset["train"].shuffle(seed=SEED).select(range(4000))
dataset["validation"] = dataset["validation"].select(range(1000))

label_names = dataset["train"].features["ner_tags"].feature.names
num_tags = len(label_names)

print(f"Label set: {label_names}")
print(f"Train size: {len(dataset['train'])}")
print(f"Val size:   {len(dataset['validation'])}")
print(f"Test size:  {len(dataset['test'])}")

# show one example
ex = dataset["train"][0]
for tok, tag_id in zip(ex["tokens"], ex["ner_tags"]):
    print(f"{tok:15s} {label_names[tag_id]}")

Label set: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']
Train size: 4000
Val size:   1000
Test size:  3453
"               O
Neither         O
the             O
National        B-MISC
Socialists      I-MISC
(               O
Nazis           B-MISC
)               O
nor             O
the             O
communists      O
dared           O
to              O
kidnap          O
an              O
American        B-MISC
citizen         O
,               O
"               O
he              O
shouted         O
,               O
in              O
an              O
oblique         O
reference       O
to              O
his             O
extradition     O
to              O
Germany         B-LOC
from            O
Denmark         B-LOC
.               O
"               O


---
# 2) Transformer-Based NER (BERT)

- Pretrained transformers like BERT (Devlin et al., 2019) provide rich contextual
representations that dramatically improve NER.
- The standard approach fine-tunes
BERT with a token classification head.

**Key challenge: subword alignment.** BERT tokenizes words into subword pieces
(e.g., "Washington" → `["Wash", "##ington"]`).

We must align word-level NER labels to subword tokens. The standard strategy:
- Assign the word's label to the **first subword**
- Assign `-100` (ignore index) to continuation subwords, `[CLS]`, `[SEP]`, `[PAD]`

This way, the cross-entropy loss is computed only on the first subword of each word.

**Sample input:** `["[CLS]", "Wash", "##ington", "is", "a", "city", "[SEP]"]`

**Sample labels:** `[-100, B-LOC, -100, O, O, O, -100]`

## 2.1 Subword-Label Alignment

Align word-level NER tags to BERT subword tokens.

- The key insight: tokenizer.word_ids() maps each subword back to its
original word index.
- We assign the label only to the first subword
of each word; all other positions get -100.

In [4]:
BERT_MODEL = "bert-base-cased"
MAX_LEN = 128
tokenizer = BertTokenizerFast.from_pretrained(BERT_MODEL)


def align_labels_to_subwords(example, tokenizer, label_names, max_len=128):
    """
    Parameters:
        example: dict with 'tokens' and 'ner_tags'
        tokenizer: BertTokenizerFast
        label_names: list of tag strings
        max_len: maximum subword sequence length

    Returns:
        tokenized: BatchEncoding with aligned 'labels'
    """
    tokenized = tokenizer(
        example["tokens"],
        is_split_into_words=True,
        padding="max_length",
        truncation=True,
        max_length=max_len,
        return_tensors="pt",
    )

    # A list that maps each subword token back to the index of the original
    # e.g. [None, 0, 1, 1, 2, 3, 3, None ...]
    word_ids = tokenized.word_ids(batch_index=0)
    tags = example["ner_tags"]
    aligned_labels = []
    prev_word_id = None

    for word_id in word_ids:
        if word_id is None:
            # special tokens: [CLS], [SEP], [PAD]
            aligned_labels.append(-100)
        elif word_id != prev_word_id:
            # first subword of a word — assign the actual label
            aligned_labels.append(tags[word_id] if word_id < len(tags) else -100)
        else:
            # continuation subword — ignore
            aligned_labels.append(-100)
        prev_word_id = word_id

    tokenized["labels"] = torch.tensor(aligned_labels, dtype=torch.long)
    return tokenized


# demonstrate alignment
demo = dataset["train"][0]
aligned = align_labels_to_subwords(demo, tokenizer, label_names)
subword_tokens = tokenizer.convert_ids_to_tokens(aligned["input_ids"].squeeze())
labels = aligned["labels"].tolist()

print("Subword alignment demo:")
for sw, lab in zip(subword_tokens[:25], labels[:25]):
    tag_str = label_names[lab] if lab != -100 else "[IGN]"
    print(f"  {sw:15s} -> {tag_str}")

Subword alignment demo:
  [CLS]           -> [IGN]
  "               -> O
  Neither         -> O
  the             -> O
  National        -> B-MISC
  Socialist       -> I-MISC
  ##s             -> [IGN]
  (               -> O
  Nazis           -> B-MISC
  )               -> O
  nor             -> O
  the             -> O
  communists      -> O
  dared           -> O
  to              -> O
  kidnap          -> O
  an              -> O
  American        -> B-MISC
  citizen         -> O
  ,               -> O
  "               -> O
  he              -> O
  shouted         -> O
  ,               -> O
  in              -> O


## 2.2 BERT NER Dataset and DataLoader

In [5]:
class BertNERDataset(Dataset):
    """Token classification dataset with subword-label alignment."""

    def __init__(self, hf_split, tokenizer, label_names, max_len=128):
        self.items = []
        for ex in hf_split:

            aligned = align_labels_to_subwords(ex, tokenizer, label_names, max_len)
            self.items.append(
                {
                    "input_ids": aligned["input_ids"].squeeze(0),
                    "attention_mask": aligned["attention_mask"].squeeze(0),
                    "labels": aligned["labels"],
                }
            )

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        return self.items[idx]


def bert_collate(batch):
    return {
        "input_ids": torch.stack([b["input_ids"] for b in batch]),
        "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
        "labels": torch.stack([b["labels"] for b in batch]),
    }


train_bert_ds = BertNERDataset(dataset["train"], tokenizer, label_names, MAX_LEN)
val_bert_ds = BertNERDataset(dataset["validation"], tokenizer, label_names, MAX_LEN)
test_bert_ds = BertNERDataset(dataset["test"], tokenizer, label_names, MAX_LEN)

train_bert_loader = DataLoader(
    train_bert_ds, batch_size=32, shuffle=True, collate_fn=bert_collate
)
val_bert_loader = DataLoader(
    val_bert_ds, batch_size=64, shuffle=False, collate_fn=bert_collate
)
test_bert_loader = DataLoader(
    test_bert_ds, batch_size=64, shuffle=False, collate_fn=bert_collate
)

print(f"Train batches: {len(train_bert_loader)}")
print(
    f"Sample batch input_ids shape: {next(iter(train_bert_loader))['input_ids'].shape}"
)

Train batches: 125
Sample batch input_ids shape: torch.Size([32, 128])


## 2.3 Architecture

In [6]:
class CustomNERModel(nn.Module):
    def __init__(self, model_name: str, num_labels: int, dropout: float = 0.1):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)
        # Grab hidden size from the model's config (so it works for any model)
        self.hidden_dim = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(p=dropout)
        self.classifier = nn.Linear(self.hidden_dim, num_labels)
        self.num_labels = num_labels

    def forward(self, input_ids, attention_mask, labels=None):
        """
        Args:
            input_ids:      (batch_num, seq_len) - Tokenized input
            attention_mask:  (batch_num, seq_len) - 1 for real tokens, 0 for padding
            labels:          (batch_num, seq_len) - Ground truth tag IDs (optional)
                            Use -100 for tokens to IGNORE in loss (e.g., [CLS], [SEP],
                            sub-word continuations, padding)

        Returns:
            dict with:
                "logits": (batch_num, seq_len, num_labels)
                "loss":   scalar (only if labels provided)
        """

        encoder_output = self.encoder(
            input_ids=input_ids, attention_mask=attention_mask
        )
        # (batch_num, seq_len, hidden_dim)
        hidden_states = encoder_output.last_hidden_state
        hidden_states = self.dropout(hidden_states)

        # (batch_num, seq_len, num_label)
        logits = self.classifier(hidden_states)

        loss = None
        if labels is not None:

            loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
            # input:  (batch_num * seq_len, num_label)
            # target: (batch_num * seq_len)
            loss = loss_fn(logits.view(-1, self.num_labels), labels.view(-1))

        return {"loss": loss, "logits": logits}

## 2.4 Fine-Tuning BERT

`CustomNERModel` above is the same design as Hugging Face's `BertForTokenClassification`: a
linear head on every token's hidden state, trained with cross-entropy that ignores positions
labelled -100.

In [7]:
model_bert = CustomNERModel(BERT_MODEL, num_labels=num_tags).to(DEVICE)

optimizer_bert = torch.optim.AdamW(model_bert.parameters(), lr=5e-5, weight_decay=0.01)

BERT_EPOCHS = 1
num_steps = len(train_bert_loader) * BERT_EPOCHS
scheduler = get_cosine_schedule_with_warmup(
    optimizer_bert,
    num_warmup_steps=int(0.1 * num_steps),
    num_training_steps=num_steps,
)

print(f"BERT parameters: {sum(p.numel() for p in model_bert.parameters()):,}")

[transformers] BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT parameters: 108,317,193


In [8]:
def evaluate_bert(model, loader, label_names):
    """Evaluate BERT NER model using seqeval entity-level F1."""
    model.eval()
    all_preds, all_trues = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)

            # (batch_num, seq_len, num_tags) -> (batch_num, seq_len)
            preds = outputs["logits"].argmax(dim=-1)

            for i in range(labels.size(0)):
                true_seq, pred_seq = [], []
                for j in range(labels.size(1)):
                    if labels[i, j].item() != -100:
                        true_seq.append(label_names[labels[i, j].item()])
                        pred_seq.append(label_names[preds[i, j].item()])
                all_trues.append(true_seq)
                all_preds.append(pred_seq)

    return f1_score(all_trues, all_preds), all_trues, all_preds


for epoch in range(BERT_EPOCHS):
    model_bert.train()
    total_loss = 0

    for batch in tqdm(train_bert_loader, desc=f"BERT Epoch {epoch+1}"):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        outputs = model_bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )
        loss = outputs["loss"]

        optimizer_bert.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_bert.parameters(), 1.0)
        optimizer_bert.step()
        scheduler.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_bert_loader)
    val_f1, _, _ = evaluate_bert(model_bert, val_bert_loader, label_names)
    print(f"  Loss: {avg_loss:.4f} | Val F1: {val_f1:.4f}")

  Loss: 0.3318 | Val F1: 0.8869


In [9]:
test_f1_bert, trues_bert, preds_bert = evaluate_bert(
    model_bert, test_bert_loader, label_names
)
print(f"\nBERT Test Entity-Level F1: {test_f1_bert:.4f}")
print(classification_report(trues_bert, preds_bert))


BERT Test Entity-Level F1: 0.8556
              precision    recall  f1-score   support

         LOC       0.83      0.90      0.86      1666
        MISC       0.77      0.64      0.70       702
         ORG       0.80      0.82      0.81      1661
         PER       0.97      0.95      0.96      1615

   micro avg       0.85      0.86      0.86      5644
   macro avg       0.84      0.83      0.83      5644
weighted avg       0.85      0.86      0.85      5644



---
# 3) GLiNER: Zero-Shot Span-Based NER


Traditional NER models are **closed-set**: they can only predict entity types seen
during training. In practice, we often need to recognize **new entity types** without
retraining — this is **zero-shot NER**.

**GLiNER** (Zaratiana et al., NAACL 2024) achieves this with a novel architecture:

<img src='https://miro.medium.com/v2/resize:fit:1400/0*_XTi6JRH-mSSRyra.png' width=800>

1. **Entity-type prompts** are prepended to the input text:
   `[ENT] person [ENT] location [SEP] Sharon Floyd flew to Miami`
2. A **bidirectional encoder** (DeBERTa) processes the combined prompt + text,
   producing representations for both entity types and words.
3. **Span representations** are computed for all possible n-gram spans in the text.
4. A **matching score** (dot product + sigmoid) is computed between each span
   representation and each entity-type representation.
5. Spans with score > threshold are predicted as entities of that type.

The key insight: entity types are defined via **natural language labels**, not
fixed class indices. At inference, you simply provide new type names.

**Sample input:** text = `"Sharon Floyd flew to Miami"`, types = `["person", "location"]`

**Sample output:** `[("Sharon Floyd", "person"), ("Miami", "location")]`

## 3.1 Span Representation Layer

Given word representations, we need to compute a fixed-size vector for every
candidate span $(i, i+k)$ 

The **SpanMarker** approach (used in GLiNER) projects start and end token
representations through separate FFNs, concatenates them, and projects back:

$$\text{span}(i, j) = W_o [\text{FFN}_{\text{start}}(h_i); \text{FFN}_{\text{end}}(h_j)]$$

In [10]:
def create_projection(input_dim: int, dropout: float, output_dim: int = None):
    """
    Two-layer FFN with ReLU activation for projecting representations.

    Architecture: Linear -> ReLU -> Dropout -> Linear
    The hidden layer uses 4x expansion following standard practice.
    """
    if output_dim is None:
        output_dim = input_dim
    return nn.Sequential(
        nn.Linear(input_dim, output_dim * 4),
        nn.ReLU(),
        nn.Dropout(dropout),
        nn.Linear(output_dim * 4, output_dim),
    )


def gather_by_index(sequence, indices):
    """
    Gather elements from a sequence tensor using index positions.

    Parameters:
        sequence: (batch_num, seq_len, model_dim)
        indices:  (batch_num, num_spans)

    Returns:
        gathered: (batch_num, num_spans, model_dim)
    """
    B, L, D = sequence.shape

    # e.g. [[0, 0], [0, 1], [1, 1], [1, 2], [2, 2], [2, 2]]
    K = indices.shape[1]

    # num_spans is usually seq_len multiplied by max_width.
    # (batch_num, num_spans, model_dim)
    expanded = indices.unsqueeze(2).expand(-1, -1, D)

    return torch.gather(sequence, 1, expanded)


class SpanMarkerRepresentation(nn.Module):
    """
    Compute span representations from start and end token embeddings.

    For each span (i, i+k), the representation is:
        span_rep = W_out(concat(FFN_start(h_i), FFN_end(h_{i+k})))

    This allows the model to learn different projections for span
    boundaries, capturing asymmetric start/end signals.
    """

    def __init__(self, model_dim: int, max_width: int, dropout: float = 0.4):
        super().__init__()
        self.max_width = max_width

        # separate projections for start and end tokens
        self.project_start = create_projection(model_dim, dropout)
        self.project_end = create_projection(model_dim, dropout)

        # final projection: concatenated start+end -> model_dim
        self.out_project = create_projection(model_dim * 2, dropout, model_dim)

    def forward(self, h, span_idx):
        """
        Parameters:
            h:        (batch_num, seq_len, model_dim) — word representations
            span_idx: (batch_num, num_spans, 2) — (start, end) indices per span

        Returns:
            span_rep: (batch_num, seq_len, max_width, model_dim)
        """
        B, L, D = h.shape

        # project all tokens through start and end FFNs
        # (batch_num, seq_len, model_dim)
        start_rep = self.project_start(h)
        end_rep = self.project_end(h)

        # if seq_len = 10, max_width = 3
        # span_idx:
        # [
        #   [0, 0], [0, 1], [0, 2],  # word 0
        #   [1, 1], [1, 2], [1, 3],  # word 1
        #   [2, 2], [2, 3], [2, 4],  # word 2
        #   ...                      # middle words
        #   [8, 8], [8, 9], [8, 9],  # word 8 (clamped to max length 9)
        # ]

        # gather start token representations for each span
        # (batch_num, num_spans, model_dim)
        start_span = gather_by_index(start_rep, span_idx[:, :, 0])

        # gather end token representations for each span
        # (batch_num, num_spans, model_dim)
        end_span = gather_by_index(end_rep, span_idx[:, :, 1])

        # concatenate start and end, apply ReLU
        # (batch_num, num_spans, model_dim * 2)
        cat = torch.cat([start_span, end_span], dim=-1).relu()

        # project back to model_dim and reshape
        # Input: (batch_num, num_spans, model_dim)
        # Output: (batch_num, seq_len, max_width, model_dim)
        return self.out_project(cat).view(B, L, self.max_width, D)


# --- Test Case for gather_by_index ---
# (batch_num, seq_len, model_dim): (1, 3, 4)
dummy_seq = torch.tensor(
    [
        [
            [0.1, 0.1, 0.1, 0.1],  # word 0
            [0.2, 0.2, 0.2, 0.2],  # word 1
            [0.3, 0.3, 0.3, 0.3],  # word 2
        ]
    ]
)

# Create dummy indices to gather from (batch=1, num_spans=2)
# gather the representation for word 0 and word 2.
dummy_idx = torch.tensor([[0, 2]])
gathered = gather_by_index(dummy_seq, dummy_idx)
print(f"Shape: {gathered.shape}")
print(f"Span: {gathered}")

Shape: torch.Size([1, 2, 4])
Span: tensor([[[0.1000, 0.1000, 0.1000, 0.1000],
         [0.3000, 0.3000, 0.3000, 0.3000]]])


## 3.2 Entity-Type Prompt Matching

The matching mechanism computes a similarity score between each span and each
entity type using a dot product. Positive spans should have high similarity
with their corresponding entity type.

Training uses **binary cross-entropy** per (span, type) pair, with negative
sampling of entity types from other examples in the batch.

In [11]:
class GLiNERModel(nn.Module):
    """
    Simplified GLiNER: entity-type prompt matching for zero-shot NER.

    Architecture:
        1. Encoder (DeBERTa) processes concatenated [prompt + text]
        2. Entity type representations extracted from [ENT] positions
        3. Word representations processed by BiLSTM for richer context
        4. Span representations computed via SpanMarker
        5. Matching score = dot product(span_rep, entity_type_rep)
    """

    def __init__(
        self,
        encoder_name: str = "microsoft/deberta-v3-small",
        model_dim: int = 768,
        max_width: int = 12,
        dropout: float = 0.4,
    ):
        super().__init__()
        self.max_width = max_width

        # special tokens for entity prompts
        self.ent_token = "[ENT]"
        self.sep_token = "[SEP]"

        # load pretrained encoder and tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(encoder_name)
        # this checkpoint is stored in fp16; the new heads below are fp32, and MPS
        # aborts on a mixed-dtype matmul, so load the encoder in fp32
        self.encoder = AutoModel.from_pretrained(encoder_name, dtype=torch.float32)

        # add special tokens
        self.tokenizer.add_tokens([self.ent_token], special_tokens=True)
        self.encoder.resize_token_embeddings(len(self.tokenizer))

        # determine encoder hidden size
        enc_dim = self.encoder.config.hidden_size

        # optional projection if encoder dim != model_dim
        if enc_dim != model_dim:
            self.projection = nn.Linear(enc_dim, model_dim)
        else:
            self.projection = nn.Identity()

        # hierarchical context via BiLSTM on word representations
        self.rnn = nn.LSTM(
            model_dim,
            model_dim // 2,
            num_layers=1,
            bidirectional=True,
            batch_first=True,
        )

        # span representation layer
        self.span_rep = SpanMarkerRepresentation(model_dim, max_width, dropout)

        # entity type representation FFN
        self.entity_type_ffn = nn.Sequential(
            nn.Linear(model_dim, model_dim * 4),
            nn.Dropout(dropout),
            nn.ReLU(),
            nn.Linear(model_dim * 4, model_dim),
        )

    def _build_prompt_input(self, tokens_batch, entity_types_batch):
        """
        Build the prompt-augmented input for the encoder.

        For each example, prepend: [ENT] type1 [ENT] type2 ... [SEP] <text tokens>

        Parameters:
            tokens_batch:       list of list of str — word-tokenized text
            entity_types_batch: list of list of str — entity type names per example

        Returns:
            all_tokens:       list of str — flat prompt+text tokens
            prompt_lengths:   list of int — length of prompt prefix per example
            num_types_list:   list of int — number of entity types per example
        """
        all_tokens = []
        prompt_lengths = []
        num_types_list = []

        for tokens, entity_types in zip(tokens_batch, entity_types_batch):
            # construct prompt: [ENT] type1 [ENT] type2 ... [SEP] token
            prompt = []
            for etype in entity_types:
                prompt.append(self.ent_token)
                prompt.append(etype)
            prompt.append(self.sep_token)

            combined = prompt + tokens
            all_tokens.append(" ".join(combined))
            prompt_lengths.append(len(prompt))
            num_types_list.append(len(entity_types))

        return all_tokens, prompt_lengths, num_types_list

    def _compute_representations(self, tokens_batch, entity_types):
        """
        Compute entity type and span representations.

        Returns:
            entity_type_reps: (batch_num, max_num_types, model_dim)
            span_reps:        (batch_num, seq_len, max_width, model_dim)
        """
        # build prompt-augmented input
        input_texts, prompt_lengths, num_types_list = self._build_prompt_input(
            tokens_batch, entity_types
        )

        # tokenize through the encoder tokenizer
        encoded = self.tokenizer(
            input_texts,
            padding=True,
            truncation=True,
            max_length=384,
            return_tensors="pt",
        ).to(next(self.parameters()).device)

        # run encoder
        # (batch_num, subword_seq_len, enc_dim)
        enc_output = self.encoder(**encoded).last_hidden_state

        # project to model_dim
        # (batch_num, subword_seq_len, model_dim)
        h = self.projection(enc_output)

        # extract entity type representations from [ENT] positions
        ent_token_id = self.tokenizer.convert_tokens_to_ids(self.ent_token)
        entity_reps_list = []
        word_reps_list = []
        word_lengths = []

        # the [ENT] tokens are at even positions in the prompt (0, 2, 4, ...)
        for i in range(len(tokens_batch)):
            input_ids_i = encoded["input_ids"][i]

            # find [ENT] token positions
            ent_positions = (input_ids_i == ent_token_id).nonzero(as_tuple=True)[0]
            ent_reps = h[i, ent_positions[: num_types_list[i]]]
            entity_reps_list.append(ent_reps)

            # find [SEP] position (first occurrence after the prompt)
            sep_id = self.tokenizer.sep_token_id
            sep_positions = (input_ids_i == sep_id).nonzero(as_tuple=True)[0]

            # word representations start after the prompt separator
            # use a simple heuristic: take the remaining tokens after prompt
            # (in production GLiNER, this uses first-subword pooling)
            if len(sep_positions) > 0:
                text_start = sep_positions[0].item() + 1
            else:
                text_start = prompt_lengths[i]

            # extract word representations (up to seq end, excluding final [SEP]/[PAD])
            attn = encoded["attention_mask"][i]
            # exclude final [SEP]
            text_end = attn.sum().item() - 1
            word_rep = h[i, text_start:text_end]
            word_reps_list.append(word_rep)
            word_lengths.append(word_rep.shape[0])

        # pad entity type representations
        # (batch_num, max_num_types, model_dim)
        entity_type_reps = pad_sequence(entity_reps_list, batch_first=True)

        # apply entity type FFN
        # (batch_num, max_num_types, model_dim)
        entity_type_reps = self.entity_type_ffn(entity_type_reps)

        # pad word representations
        # (batch_num, max_seq_len, model_dim)
        word_reps = pad_sequence(word_reps_list, batch_first=True)

        # build mask for BiLSTM
        max_wlen = word_reps.shape[1]
        word_mask = torch.arange(max_wlen, device=word_reps.device).unsqueeze(
            0
        ) < torch.tensor(word_lengths, device=word_reps.device).unsqueeze(1)

        # apply BiLSTM for hierarchical encoding
        lengths_cpu = torch.tensor(word_lengths)
        lengths_cpu = lengths_cpu.clamp(min=1)  # avoid zero-length
        packed = pack_padded_sequence(
            word_reps, lengths_cpu.cpu(), batch_first=True, enforce_sorted=False
        )
        rnn_out, _ = self.rnn(packed)

        # (batch_num, max_seq_len, model_dim)
        word_reps, _ = pad_packed_sequence(rnn_out, batch_first=True)

        # build span indices: for each position i, spans (i, i), (i, i+1), ..., (i, i+max_width-1)
        seq_len = word_reps.shape[1]
        span_indices = []
        for i in range(seq_len):
            for k in range(self.max_width):
                # clamp end index to valid range
                span_indices.append([i, min(i + k, seq_len - 1)])

        # (1, num_spans, 2) -> (batch_num, num_spans, 2)
        span_idx = torch.tensor(span_indices, device=word_reps.device).unsqueeze(0)
        span_idx = span_idx.expand(word_reps.shape[0], -1, -1)

        # compute span representations
        # (batch_num, seq_len, max_width, model_dim)
        span_representations = self.span_rep(word_reps, span_idx)

        return entity_type_reps, span_representations, word_lengths

    def compute_scores(self, tokens_batch, entity_types):
        """
        Compute matching scores between spans and entity types.

        Returns:
            scores: (batch_num, seq_len, max_width, num_types)
        """
        entity_reps, span_reps, word_lengths = self._compute_representations(
            tokens_batch, entity_types
        )

        # dot product matching
        # span_reps: (batch_num, seq_len, max_width, model_dim)
        # entity_reps: (batch_num, num_types, model_dim)
        # scores: (batch_num, seq_len, max_width, num_types)
        scores = torch.einsum("BLKD,BCD->BLKC", span_reps, entity_reps)
        return scores, word_lengths

In [12]:
# Smoke test of the from-scratch model (untrained heads, so only the shapes mean anything)
toy_gliner = GLiNERModel().to(DEVICE).eval()
with torch.no_grad():
    scores, lengths = toy_gliner.compute_scores(
        [["Sharon", "Floyd", "flew", "to", "Miami"]], [["person", "location"]]
    )
# (batch_num, num_subword_positions, max_width, num_types)
print(f"Span-type scores: {tuple(scores.shape)}, text subwords: {lengths}")
del toy_gliner

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Span-type scores: (1, 5, 12, 2), text subwords: [5]


## 3.3 Using Pre-Trained GLiNER

Training GLiNER from scratch requires the Pile-NER dataset (250K+ examples)
and significant compute. For practical use, we load a pretrained model
from Hugging Face and demonstrate zero-shot NER on arbitrary entity types.

The `gliner` library wraps the architecture above with optimized inference,
including greedy span decoding and configurable thresholds.

Available models (as of 2026):
- `urchade/gliner_small-v2.1` — 166M params, Apache 2.0
- `urchade/gliner_medium-v2.1` — 209M params, Apache 2.0
- `knowledgator/gliner-bi-base-v2.0` — bi-encoder, scales to 1000+ entity types

In [13]:
from gliner import GLiNER

gliner_model = GLiNER.from_pretrained("urchade/gliner_small-v2.1")
gliner_model = gliner_model.to(DEVICE)
gliner_model.eval()
# ; hides the full module printout

UniEncoderSpanGLiNER(
  (model): UniEncoderSpanModel(
    (token_rep_layer): Encoder(
      (bert_layer): Transformer(
        (model): DebertaV2Model(
          (embeddings): DebertaV2Embeddings(
            (word_embeddings): Embedding(128004, 768, padding_idx=0)
            (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (encoder): DebertaV2Encoder(
            (layer): ModuleList(
              (0-5): 6 x DebertaV2Layer(
                (attention): DebertaV2Attention(
                  (self): DisentangledSelfAttention(
                    (query_proj): Linear(in_features=768, out_features=768, bias=True)
                    (key_proj): Linear(in_features=768, out_features=768, bias=True)
                    (value_proj): Linear(in_features=768, out_features=768, bias=True)
                    (pos_dropout): Dropout(p=0.1, inplace=False)
                    (dropout): Dropout(p

In [14]:
# zero-shot NER with arbitrary entity types
text = (
    "Peter Navarro, the White House director of trade and manufacturing policy, "
    "said in an interview on Sunday morning that the administration was preparing "
    "for a possible second wave of COVID-19 in the fall."
)

# standard entity types
labels_standard = ["person", "organization", "location", "date", "event"]
entities = gliner_model.predict_entities(text, labels_standard, threshold=0.5)

print("=== Standard entity types ===")
for e in entities:
    print(f"  {e['text']:30s}  {e['label']:15s}  (score: {e['score']:.3f})")

=== Standard entity types ===
  Peter Navarro                   person           (score: 0.988)
  White House                     organization     (score: 0.857)
  Sunday morning                  date             (score: 0.520)
  administration                  organization     (score: 0.658)
  COVID-19                        event            (score: 0.523)
  fall                            date             (score: 0.901)


In [15]:
# zero-shot with NOVEL entity types — no retraining needed
labels_custom = ["government role", "policy area", "disease", "time period"]
entities_custom = gliner_model.predict_entities(text, labels_custom, threshold=0.4)

print("=== Custom entity types (zero-shot) ===")
for e in entities_custom:
    print(f"  {e['text']:40s}  {e['label']:20s}  (score: {e['score']:.3f})")

=== Custom entity types (zero-shot) ===
  trade and manufacturing policy            policy area           (score: 0.533)
  COVID-19                                  disease               (score: 0.969)
  fall                                      time period           (score: 0.873)


In [16]:
# evaluate GLiNER on CoNLL-2003 test set (zero-shot)
conll_types = ["person", "organization", "location", "miscellaneous"]

# mapping from GLiNER labels back to CoNLL BIO tags
type_to_bio = {
    "person": "PER",
    "organization": "ORG",
    "location": "LOC",
    "miscellaneous": "MISC",
}

# evaluate on subset for speed
all_true_gliner, all_pred_gliner = [], []
num_eval = min(500, len(dataset["test"]))

for idx in tqdm(range(num_eval), desc="GLiNER eval"):
    ex = dataset["test"][idx]
    tokens = ex["tokens"]
    true_tags = [label_names[t] for t in ex["ner_tags"]]

    sentence = " ".join(tokens)
    preds = gliner_model.predict_entities(sentence, conll_types, threshold=0.5)

    # character start of every token in the joined sentence
    starts, pos = [], 0
    for tok in tokens:
        starts.append(pos)
        pos += len(tok) + 1

    # convert span predictions to BIO token labels via GLiNER's character offsets,
    # so a name that appears twice is tagged at the right occurrence
    pred_tags = ["O"] * len(tokens)
    for ent in preds:
        bio_prefix = type_to_bio[ent["label"]]
        inside = [i for i, s in enumerate(starts) if ent["start"] <= s < ent["end"]]
        for k, i in enumerate(inside):
            pred_tags[i] = f"{'B' if k == 0 else 'I'}-{bio_prefix}"

    all_true_gliner.append(true_tags)
    all_pred_gliner.append(pred_tags)

gliner_f1 = f1_score(all_true_gliner, all_pred_gliner)
print(
    f"\nGLiNER Zero-Shot F1 on CoNLL-2003 (first {num_eval} examples): {gliner_f1:.4f}"
)
# "miscellaneous" is a catch-all, not a description, so a zero-shot type matcher cannot use it
print(classification_report(all_true_gliner, all_pred_gliner))


GLiNER Zero-Shot F1 on CoNLL-2003 (first 500 examples): 0.6970
              precision    recall  f1-score   support

         LOC       0.58      0.79      0.67       288
        MISC       0.00      0.00      0.00        65
         ORG       0.36      0.29      0.32       188
         PER       0.95      0.92      0.93       442

   micro avg       0.69      0.70      0.70       983
   macro avg       0.47      0.50      0.48       983
weighted avg       0.66      0.70      0.68       983

